In [ ]:
import numpy as np
import pandas as pd
import yaml
from haversine import haversine, Unit
import ast
from sklearn.preprocessing import LabelEncoder

with open("config.yaml", "r") as f:
    config = yaml.safe_load(f)

DATA_URL = config["data_url"]

LONLAT_MAPPING_DICT = {
    "uncle kkm (bukit panjang)": (1.3770731732807227, 103.76455875389925),
    "bbdc": (1.367094085362527, 103.75025701156822),
    "raffles place": (1.2844344943505839, 103.84979077988179),
    "cck": (1.3849417640590647, 103.74457971156812),
    "suntec": (1.2952635056725392, 103.85829414863679),
}

def int_to_datetime(val, date):
    if pd.isna(val):
        return pd.NaT
    val = int(val)
    hours = val // 100
    minutes = val % 100
    return pd.Timestamp(f"{date.date()} {hours:02d}:{minutes:02d}")

def calc_distance(row, origin_col, destination_col):
    try:
        origin = ast.literal_eval(row[origin_col]) if isinstance(row[origin_col], str) else row[origin_col]
        destination = ast.literal_eval(row[destination_col]) if isinstance(row[destination_col], str) else row[destination_col]
        return haversine(origin, destination, unit=Unit.KILOMETERS)
    except (ValueError, TypeError):
        return None

### Import Data

In [2]:
# Read data from Google Excel Sheet
df = pd.read_csv(DATA_URL)

df

,Date,Time to meet,Actual time arrived,Late flag,Location,Category,home_latlon
0,1/2/2026,1100,1107.0,True,Uncle KKM (bukit panjang),Lunch,"(1.3824797878551964, 103.75444675699774)"
1,1/3/2026,915,920.0,False,Virtual,Apply job,"(1.3824797878551964, 103.75444675699774)"
2,1/6/2026,1930,2005.0,True,BBDC,Exercise,"(1.3824797878551964, 103.75444675699774)"
3,1/8/2026,1900,1948.0,True,Raffles Place,Dinner/drinks,"(1.3824797878551964, 103.75444675699774)"
4,1/15/2026,1900,NaN,True,IMM,Dinner/drinks,"(1.3824797878551964, 103.75444675699774)"
5,1/23/2026,2000,2100.0,True,CCK,Dinner/drinks,"(1.3824797878551964, 103.75444675699774)"
6,3/8/2026,920,930.0,True,Uncle KKM (bukit panjang),Breakfast,"(1.3824797878551964, 103.75444675699774)"
7,3/27/2026,2000,2015.0,True,Her house,Dinner/drinks,"(1.3824797878551964, 103.75444675699774)"
8,3/28/2026,1500,1530.0,True,Suntec,work/career fair,"(1.3824797878551964, 103.75444675699774)"
9,4/10/2026,1830,1839.0,NaN,Suntec,Dinner/drinks,"(1.3824797878551964, 103.75444675699774)"


### Data Cleaning and Transformation

In [3]:
# Drop late flag column since it is not accurate and rename columns
modified_df = (
    df[["Date", "Location", "Category", "Time to meet", "Actual time arrived", "home_latlon"]]
    .rename(
        columns={
            "Date": "date",
            "Time to meet": "meeting_time",
            "Actual time arrived": "arrived_time",
            "Location": "meeting_location",
            "Category": "category"
        }
    )
)

# Convert date to date values
modified_df["date"] = pd.to_datetime(modified_df["date"])
modified_df = modified_df.sort_values(by="date")

# Acquire date features
modified_df["month"] = modified_df["date"].dt.month
modified_df["day_of_week"] = modified_df["date"].dt.dayofweek

# Convert meeting and arrival times to datetime values
modified_df["meeting_time"] = modified_df.apply(lambda row: int_to_datetime(row["meeting_time"], row["date"]), axis=1)
modified_df["arrived_time"] = modified_df.apply(lambda row: int_to_datetime(row["arrived_time"], row["date"]), axis=1)

# Lowercase and clear white spacing
modified_df["meeting_location"] = modified_df["meeting_location"].str.strip().str.lower()
modified_df["category"] = modified_df["category"].str.strip().str.lower()

# Calculate how late in minutes
modified_df["late_duration_min"] = (modified_df["arrived_time"] - modified_df["meeting_time"]).dt.total_seconds() / 60

# Drop rows where late_duration_min is NaN
modified_df = modified_df.dropna(subset="late_duration_min")

# Drop rows that have virtual meeting since only interested in physical
modified_df = modified_df[~modified_df["meeting_location"].isin(["virtual", "her house"])]

# Convert meeting location into latlon
modified_df["meeting_latlon"] = modified_df["meeting_location"].str.lower().str.strip().map(LONLAT_MAPPING_DICT)

# Calculate the distance away from home to meeting location
modified_df["distance_km"] = (
    modified_df.apply(
        calc_distance,
        axis=1,
        origin_col="home_latlon",
        destination_col="meeting_latlon"
    )
)

modified_df

,date,meeting_location,category,meeting_time,arrived_time,home_latlon,month,day_of_week,late_duration_min,meeting_latlon,distance_km
0,2026-01-02,uncle kkm (bukit panjang),lunch,2026-01-02 11:00:00,2026-01-02 11:07:00,"(1.3824797878551964, 103.75444675699774)",1,4,7.0,"(1.3770731732807227, 103.76455875389925)",1.274747
2,2026-01-06,bbdc,exercise,2026-01-06 19:30:00,2026-01-06 20:05:00,"(1.3824797878551964, 103.75444675699774)",1,1,35.0,"(1.367094085362527, 103.75025701156822)",1.773078
3,2026-01-08,raffles place,dinner/drinks,2026-01-08 19:00:00,2026-01-08 19:48:00,"(1.3824797878551964, 103.75444675699774)",1,3,48.0,"(1.2844344943505839, 103.84979077988179)",15.205063
5,2026-01-23,cck,dinner/drinks,2026-01-23 20:00:00,2026-01-23 21:00:00,"(1.3824797878551964, 103.75444675699774)",1,4,60.0,"(1.3849417640590647, 103.74457971156812)",1.130494
6,2026-03-08,uncle kkm (bukit panjang),breakfast,2026-03-08 09:20:00,2026-03-08 09:30:00,"(1.3824797878551964, 103.75444675699774)",3,6,10.0,"(1.3770731732807227, 103.76455875389925)",1.274747
8,2026-03-28,suntec,work/career fair,2026-03-28 15:00:00,2026-03-28 15:30:00,"(1.3824797878551964, 103.75444675699774)",3,5,30.0,"(1.2952635056725392, 103.85829414863679)",15.077114
9,2026-04-10,suntec,dinner/drinks,2026-04-10 18:30:00,2026-04-10 18:39:00,"(1.3824797878551964, 103.75444675699774)",4,4,9.0,"(1.2952635056725392, 103.85829414863679)",15.077114
10,2026-04-17,uncle kkm (bukit panjang),lunch,2026-04-17 12:45:00,2026-04-17 12:58:00,"(1.3824797878551964, 103.75444675699774)",4,4,13.0,"(1.3770731732807227, 103.76455875389925)",1.274747


In [4]:
# Filter for feature columns
feature_df = (
    modified_df[["day_of_week", "distance_km", "category", "late_duration_min"]]
    .reset_index(drop=True)
)

feature_df

,day_of_week,distance_km,category,late_duration_min
0,4,1.274747,lunch,7.0
1,1,1.773078,exercise,35.0
2,3,15.205063,dinner/drinks,48.0
3,4,1.130494,dinner/drinks,60.0
4,6,1.274747,breakfast,10.0
5,5,15.077114,work/career fair,30.0
6,4,15.077114,dinner/drinks,9.0
7,4,1.274747,lunch,13.0


### Export Feature Data

In [ ]:
feature_df.to_parquet("data/feature_df.parquet")